# Pregel Super-Steps & Frontier Logic

A self-contained reference notebook explaining the execution model behind `tinygraph` (and LangGraph).

**Core idea**: instead of a sequential call stack, a graph executes in *waves*.  
Each wave — called a **super-step** — runs all nodes that are ready at the same time.  
After each wave, state is merged, and the next wave is computed.

---

## 1. Why not just call nodes in order?

For a linear chain `A → B → C` you could. But consider:

```
START → A ─┐
           ├→ C → END
START → B ─┘
```

- **A and B are independent** — they can run at the same time.
- **C depends on both** — it must wait for A *and* B to finish.

Sequential order can't express this. Super-steps can.

## 2. Building blocks

Everything we need: a graph (dict of adjacency sets) and its inverse (predecessors).

In [ ]:
from collections import defaultdict


def build_predecessors(graph: dict[str, set[str]]) -> dict[str, set[str]]:
    """Invert an adjacency dict: for each node, who points to it?"""
    pred: dict[str, set[str]] = defaultdict(set)
    for source, targets in graph.items():
        for target in targets:
            pred[target].add(source)
    return dict(pred)


# Example graph: START → {A, B}, A → C, B → C, C → END
graph = {
    "START": {"A", "B"},
    "A": {"C"},
    "B": {"C"},
    "C": {"END"},
}

predecessors = build_predecessors(graph)
print("Predecessors:")
for node, preds in sorted(predecessors.items()):
    print(f"  {node}: {preds}")

Expected output:
```
Predecessors:
  A:   {'START'}
  B:   {'START'}
  C:   {'A', 'B'}   ← two parents, fan-in point
  END: {'C'}
```

## 3. The barrier condition

A node `n` is **ready** when all of its *relevant* parents have finished.

```python
(predecessors.get(n, set()) & barrier_scope) <= completed
```

Three pieces:

| Expression | Meaning |
|---|---|
| `predecessors.get(n, set())` | All parents of `n` in the graph structure |
| `& barrier_scope` | Keep only parents that are *relevant* to this execution |
| `<= completed` | All of those parents must be done |

`barrier_scope = activated | candidates`  
= every node we have *decided to run* at some point during this execution.

In [ ]:
def is_ready(
    node: str,
    predecessors: dict[str, set[str]],
    barrier_scope: set[str],
    completed: set[str],
) -> bool:
    """Is `node` ready to enter the next frontier?"""
    relevant_parents = predecessors.get(node, set()) & barrier_scope
    return relevant_parents <= completed


# Manual check: is C ready when only A is done?
completed = {"START", "A"}
activated = {"START", "A", "B"}  # B is activated but not yet completed
candidates = {"C"}  # A's successors
barrier_scope = activated | candidates

print("C ready when only A done?", is_ready("C", predecessors, barrier_scope, completed))
# Expected: False — B is in barrier_scope but not in completed

# Now B finishes too
completed = {"START", "A", "B"}
print("C ready when both A and B done?", is_ready("C", predecessors, barrier_scope, completed))
# Expected: True

## 4. `completed` vs `activated` — why two sets?

| Set | Grows when | Represents |
|---|---|---|
| `activated` | A node enters a frontier | *"has been put in play"* |
| `completed` | A node finishes executing | *"has actually run"* |

`activated` is always ≥ `completed`.  
The gap between them = nodes currently running (the current frontier).

**Why `activated` matters**: without it, nodes running in the *current* frontier  
would be invisible to the barrier check, and their successors could be released too early.

In [ ]:
# Demonstrate the problem without activated

# Suppose we use only completed for barrier_scope:
# At the START of super-step 1: completed={START}, candidates={A,B}

bad_barrier_scope = {"START"} | {"A", "B", "C"}  # completed | candidates (no activated)
good_barrier_scope = {"START", "A", "B"} | {"C"}  # activated | candidates

completed_at_start_of_step1 = {"START"}

# Can C run at the very start, before A and B have run?
bad_result = is_ready("C", predecessors, bad_barrier_scope, completed_at_start_of_step1)
good_result = is_ready("C", predecessors, good_barrier_scope, completed_at_start_of_step1)

print(f"Without activated — C ready at start of step 1? {bad_result}")
# True — WRONG: C would run before A and B

print(f"With activated    — C ready at start of step 1? {good_result}")
# False — CORRECT: C waits for A and B

## 5. Why `& barrier_scope` is necessary — conditional edges

Consider a graph where a router chooses **one** branch at runtime:

```
START → A → (router chooses B or C, never both)
B → END
C → END
```

`predecessors[END] = {"B", "C"}`

If the router picks B, C is never activated.  
Without `& barrier_scope`, the barrier waits for C forever → **graph stuck**.

In [ ]:
# Conditional graph: A routes to B only (C is never activated)
cond_graph = {
    "START": {"A"},
    "A": {"B"},  # at runtime, router picked B
    "B": {"END"},
    # C also points to END in the schema, but A never activated it
}

cond_predecessors = build_predecessors(cond_graph)
# But END's structural predecessors (from schema) include both B and C:
cond_predecessors["END"] = {"B", "C"}  # simulate schema knowledge

print("END's structural predecessors:", cond_predecessors["END"])

# After B runs:
completed = {"START", "A", "B"}
activated = {"START", "A", "B", "END"}  # C was never activated
candidates = {"END"}
barrier_scope = activated | candidates

print("\nWith & barrier_scope:")
relevant = cond_predecessors["END"] & barrier_scope
print(f"  relevant parents of END: {relevant}")
print(f"  {relevant} <= {completed} ? {relevant <= completed}")
# {B} <= {START,A,B} → True ✓ — END is released

print("\nWithout & barrier_scope:")
all_parents = cond_predecessors["END"]
print(f"  all parents of END: {all_parents}")
print(f"  {all_parents} <= {completed} ? {all_parents <= completed}")
# {B,C} <= {START,A,B} → False ✗ — graph stuck forever

## 6. Full simulation — step by step

Now we put it all together: a complete super-step loop with tracing.

In [ ]:
def get_candidates(frontier: set[str], graph: dict[str, set[str]]) -> set[str]:
    """All successor nodes of the current frontier."""
    result: set[str] = set()
    for node in frontier:
        result |= graph.get(node, set())
    return result


def next_frontier(
    candidates: set[str],
    completed: set[str],
    activated: set[str],
    predecessors: dict[str, set[str]],
) -> set[str]:
    """Filter candidates down to those whose relevant parents are all done."""
    barrier_scope = activated | candidates
    return {n for n in candidates if (predecessors.get(n, set()) & barrier_scope) <= completed}


def simulate(graph: dict[str, set[str]], max_steps: int = 10) -> None:
    """Run the super-step loop and print each step."""
    preds = build_predecessors(graph)

    completed: set[str] = {"START"}
    activated: set[str] = {"START"}

    candidates = get_candidates({"START"}, graph)
    frontier = next_frontier(candidates, completed, activated, preds)
    activated |= frontier

    for step in range(1, max_steps + 1):
        if "END" in frontier:
            print(f"Step {step}: END reached — done.")
            break

        print(f"Step {step}: execute {sorted(frontier)}")

        # Nodes finish executing
        completed |= frontier

        # Compute next frontier
        candidates = get_candidates(frontier, graph)
        frontier = next_frontier(candidates, completed, activated, preds)
        activated |= frontier

        if not frontier:
            print("  !! No frontier and END not reached — graph is stuck.")
            break
    else:
        print(f"!! Exceeded {max_steps} steps — possible cycle.")

In [ ]:
# ── Example 1: linear chain ──────────────────────────────────────────────────
print("=== Linear: START → A → B → C → END ===")
simulate(
    {
        "START": {"A"},
        "A": {"B"},
        "B": {"C"},
        "C": {"END"},
    }
)

In [ ]:
# ── Example 2: fan-out / fan-in ──────────────────────────────────────────────
print("=== Fan-out/fan-in: START → {A,B} → C → END ===")
simulate(
    {
        "START": {"A", "B"},
        "A": {"C"},
        "B": {"C"},
        "C": {"END"},
    }
)

In [ ]:
# ── Example 3: diamond (wider fan) ──────────────────────────────────────────
print("=== Diamond: START → A → {B,C} → D → END ===")
simulate(
    {
        "START": {"A"},
        "A": {"B", "C"},
        "B": {"D"},
        "C": {"D"},
        "D": {"END"},
    }
)

In [ ]:
# ── Example 4: two independent chains merging ───────────────────────────────
print("=== Dual chain: START → {A,C}, A→B, C→D, {B,D} → END ===")
simulate(
    {
        "START": {"A", "C"},
        "A": {"B"},
        "B": {"END"},
        "C": {"D"},
        "D": {"END"},
    }
)

## 7. Quick reference

```
frontier    = nodes executing in this super-step
activated   = all nodes that have been placed in any frontier (ever)
completed   = all nodes that have finished executing
candidates  = successors of the current frontier (wants to run next)
barrier_scope = activated | candidates
```

**A candidate node enters the next frontier when:**
```
(predecessors[n] & barrier_scope) ⊆ completed
```
= *all relevant parents are done*

**Why `& barrier_scope`?**  
Filters out parents that were never activated (conditional branches not taken).  
Without this filter, the graph stalls forever waiting for a branch that will never run.

**Why `activated` instead of just `completed`?**  
Nodes in the *current* frontier are activated but not yet completed.  
Including them in `barrier_scope` prevents their successors from being released too early.

---
*Part of the `tinygraph` educational series — Phase 2: rebuilding LangGraph primitives from scratch.*